<a href="https://colab.research.google.com/github/dohee-jin/data-mining-final-project/blob/main/kbo_goldenglove_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 골든글러브 후보 예측

- 선수 개인 기록을 활용해 2026년 골든글러브 후보 가능성이 높은 선수를 예측한다.
- 타자와 투수는 기록 컬럼이 다르므로 분리해서 분석한다.

---

## 2. 골든글로브 후보 예측 데이터 사용 전략

데이터는 크게 세 종류로 나눈다.

| 구분 | 출처 | 직접 정리 여부 | 사용 목적 |
|---|---|---:|---|
| 과거 선수 기록 | Kaggle KBO Player Dataset 1982-2025 | X | 골든글러브 예측 학습 데이터 |
| 2026 현재 선수 기록 | KBO 공식 홈페이지 기록실 | O | 2026 골든글러브 후보 예측 대상 |
| 골든글러브 수상자 목록 | KBO 공식 골든글러브 수상 현황 | O | `is_goldenglove` 라벨 생성 |

---

## 3. 골든글러브 후보 예측용 데이터

### 3.1 Kaggle에서 가져올 데이터

Kaggle의 `KBO Player Dataset (1982-2025)`를 과거 학습 데이터로 사용한다.

사용할 데이터는 다음과 같다.

```text
kbo_batting_stats_by_season_1982-2025.csv
kbo_pitching_stats_by_season_1982-2025.csv
```

Kaggle 데이터는 1982년부터 2025년까지의 KBO 정규시즌 선수 타격 및 투수 기록을 포함한다. 실제 분석에서는 너무 오래된 데이터까지 모두 쓰기보다, 최근 야구 환경과 비슷한 기간만 필터링하여 사용한다.

추천 사용 기간:

```text
2015~2025
```

### 3.2 직접 정리해야 하는 2026 선수 기록

2026년은 아직 시즌이 진행 중이므로 Kaggle 과거 데이터에 포함되어 있지 않다. 따라서 KBO 공식 홈페이지에서 현재 기록을 복사해 CSV로 직접 정리하고 깃허브에 csv로 raw 파일을 업로드했다.

```text
data/raw/kbo_hitter_data_2026.csv
data/raw/kbo_pitcher_data_2026.csv
```

2026 선수 기록 파일에는 반드시 `year` 컬럼을 추가한다.   
타자 기록에는 골든글러브 포지션별 예측을 위해 `position` 칼럼을 추가한다.   
일부 선수는 시즌 중 여러 포지션에 출전하므로, 본 분석에서는 데이터 단순화와 모델 적용의 일관성을 위해 선수별 대표 포지션을 하나만 부여하였다. 대표 포지션은 KBO 기록실의 선수 구분 또는 주요 출전 포지션을 기준으로 정리하였다.

```text
year = 2026
position = C, 1B, 2B, 3B, SS, OF, DH
```

예시:

| year | player | team | avg | hr | rbi | ops |
|---:|---|---|---:|---:|---:|---:|
| 2026 | 선수명 | LG | 0.315 | 12 | 45 | 0.890 |

---

### 4. kaggle 데이터셋 접근 불가
kaggle 데이터셋이 내려가 접근하지 못해 해당 데이터셋으로의 모델 생성, 예측은 불가능하게 됨.  
`https://huggingface.co/datasets/juhonov/KBOresearch?utm_source=chatgpt.com` 의 선수 기록 수집기를 통해 2000-2025년도 선수 기록을 바탕으로 데이터를 수집, 학습하기로 계획을 변경함

### kagglehub로 kbo 1982-2025 데이터셋 불러오기
kaggle내 데이터셋이 삭제되어 사용불가

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("netsong/kbo-player-dataset-by-regular-season-1982-2025")

print("Path to dataset files:", path)

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

In [ ]:
import os
import pandas as pd

path = kagglehub.dataset_download("netsong/kbo-player-dataset-by-regular-season-1982-2025")

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

## 다른 데이터셋 사용

In [ ]:
!pip install -q datasets

from datasets import load_dataset
from huggingface_hub import hf_hub_download
import pandas as pd

# Hugging Face에서 JSON 파일 직접 다운로드
hitter_file = hf_hub_download(
    repo_id="juhonov/KBOresearch",
    filename="kbo_hitter_stats_2000_2025.json",
    repo_type="dataset"
)

pitcher_file = hf_hub_download(
    repo_id="juhonov/KBOresearch",
    filename="kbo_pitcher_stats_2000_2025.json",
    repo_type="dataset"
)

print(hitter_file)
print(pitcher_file)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


kbo_hitter_stats_2000_2025.json:   0%|          | 0.00/3.44M [00:00<?, ?B/s]

kbo_pitcher_stats_2000_2025.json:   0%|          | 0.00/2.74M [00:00<?, ?B/s]

/root/.cache/huggingface/hub/datasets--juhonov--KBOresearch/snapshots/14fd8e53426a5e3bd21b576fa109ea8928463008/kbo_hitter_stats_2000_2025.json
/root/.cache/huggingface/hub/datasets--juhonov--KBOresearch/snapshots/14fd8e53426a5e3bd21b576fa109ea8928463008/kbo_pitcher_stats_2000_2025.json


In [ ]:
# Json 구조 확인
import json

with open(hitter_file, "r", encoding="utf-8") as f:
    hitter_json = json.load(f)

with open(pitcher_file, "r", encoding="utf-8") as f:
    pitcher_json = json.load(f)

print(type(hitter_json))
print(type(pitcher_json))

if isinstance(hitter_json, dict):
    print(hitter_json.keys())

if isinstance(pitcher_json, dict):
    print(pitcher_json.keys())

<class 'dict'>
<class 'dict'>
dict_keys(['metadata', 'data'])
dict_keys(['metadata', 'data'])


In [ ]:
# json 데이터 구조 확인
def inspect_json(obj, name="root"):
    print(f"\n{name}: type={type(obj)}")

    if isinstance(obj, dict):
        print("keys:", list(obj.keys()))
        for k, v in obj.items():
            print(f"  {k}: {type(v)}", end="")
            if isinstance(v, list):
                print(f", list length={len(v)}")
                if len(v) > 0:
                    print("    first item type:", type(v[0]))
                    print("    first item:", v[0])
            elif isinstance(v, dict):
                print(f", dict keys={list(v.keys())[:10]}")
            else:
                print(f", value={v}")

    elif isinstance(obj, list):
        print("list length:", len(obj))
        if len(obj) > 0:
            print("first item type:", type(obj[0]))
            print("first item:", obj[0])

inspect_json(hitter_json, "hitter_json")
inspect_json(pitcher_json, "pitcher_json")


hitter_json: type=<class 'dict'>
keys: ['metadata', 'data']
  metadata: <class 'dict'>, dict keys=['description', 'source', 'collected_at', 'total_records']
  data: <class 'list'>, list length=8188
    first item type: <class 'dict'>
    first item: {'순위': '1', 'playerId': '62558', '선수명': '김준태', '팀명': 'LG', 'AVG': '1.000', 'G': '2', 'PA': '2', 'AB': '1', 'R': '1', 'H': '1', '2B': '0', '3B': '0', 'HR': '0', 'TB': '1', 'RBI': '0', 'SAC': '0', 'SF': '0', 'year': 2025, 'teamCode': 'LG', 'teamName': 'LG'}

pitcher_json: type=<class 'dict'>
keys: ['metadata', 'data']
  metadata: <class 'dict'>, dict keys=['description', 'source', 'collected_at', 'total_records']
  data: <class 'list'>, list length=5696
    first item type: <class 'dict'>
    first item: {'순위': '1', 'playerId': '50904', '선수명': '이종준', '팀명': 'LG', 'ERA': '0.00', 'G': '2', 'W': '0', 'L': '0', 'SV': '0', 'HLD': '0', 'WPCT': '-', 'IP': '1 2/3', 'H': '0', 'HR': '0', 'BB': '1', 'HBP': '0', 'SO': '2', 'R': '0', 'ER': '0', 'WHIP': '0

In [ ]:
from google.colab import files

# 데이터프레임 생성
hitter = pd.DataFrame(hitter_json["data"])
pitcher = pd.DataFrame(pitcher_json["data"])

print(hitter.shape)
print(pitcher.shape)

display(hitter.head())
display(pitcher.head())

# 데이터프레임 로컬 저장
# 로컬 저장 후 깃허브에 직접 업로드 필요
"""
hitter.to_csv("kbo_hitter_stats_2000-2025.csv", index=False, encoding="utf-8-sig")
pitcher.to_csv("kbo_pitcher_stats_2000-2025.csv", index=False, encoding="utf-8-sig")

files.download("kbo_hitter_stats_2000-2025.csv")
files.download("kbo_pitcher_stats_2000-2025.csv")
"""

(8188, 27)
(5696, 26)


,순위,playerId,선수명,팀명,AVG,G,PA,AB,R,H,...,year,teamCode,teamName,SB,CS,BB,HBP,SO,GDP,E
0,1,62558,김준태,LG,1.000,2,2,1,1,1,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,54166,김현종,LG,0.400,10,6,5,3,2,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,65207,신민재,LG,0.313,135,538,463,87,145,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,53123,오스틴,LG,0.313,116,499,425,82,133,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,68119,문성주,LG,0.305,135,542,475,57,145,...,2025,LG,LG,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,순위,playerId,선수명,팀명,ERA,G,W,L,SV,HLD,...,SO,R,ER,WHIP,year,teamCode,teamName,CG,SHO,TBF
0,1,50904,이종준,LG,0.00,2,0,0,0,0,...,2,0,0,0.60,2025,LG,LG,NaN,NaN,NaN
1,2,77263,김강률,LG,1.46,12,1,0,1,4,...,9,2,2,1.22,2025,LG,LG,NaN,NaN,NaN
2,3,61145,이우찬,LG,1.89,23,0,1,0,0,...,20,6,4,1.42,2025,LG,LG,NaN,NaN,NaN
3,4,55167,김영우,LG,2.40,66,3,2,1,7,...,56,17,16,1.32,2025,LG,LG,NaN,NaN,NaN
4,5,50106,유영찬,LG,2.63,39,2,2,21,1,...,52,14,12,1.32,2025,LG,LG,NaN,NaN,NaN


'\nhitter.to_csv("kbo_hitter_stats_2000-2025.csv", index=False, encoding="utf-8-sig")\npitcher.to_csv("kbo_pitcher_stats_2000-2025.csv", index=False, encoding="utf-8-sig")\n\nfiles.download("kbo_hitter_stats_2000-2025.csv")\nfiles.download("kbo_pitcher_stats_2000-2025.csv")\n'

In [ ]:
# hitter = pd.read_csv("/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_batting_stats_by_season_1982-2025.csv")
# pitcher = pd.read_csv("/kaggle/input/kbo-player-dataset-by-regular-season-1982-2025/kbo_pitching_stats_by_season_1982-2025.csv")

hitter.head()
pitcher.head()

# 2015~2025년 데이터만 필터링
hitter_recent = hitter[hitter["year"] > 2015]
pitcher_recent = pitcher[pitcher["year"] > 2015]

hitter_recent.head()
pitcher_recent.head()

,순위,playerId,선수명,팀명,ERA,G,W,L,SV,HLD,...,SO,R,ER,WHIP,year,teamCode,teamName,CG,SHO,TBF
0,1,50904,이종준,LG,0.00,2,0,0,0,0,...,2,0,0,0.60,2025,LG,LG,NaN,NaN,NaN
1,2,77263,김강률,LG,1.46,12,1,0,1,4,...,9,2,2,1.22,2025,LG,LG,NaN,NaN,NaN
2,3,61145,이우찬,LG,1.89,23,0,1,0,0,...,20,6,4,1.42,2025,LG,LG,NaN,NaN,NaN
3,4,55167,김영우,LG,2.40,66,3,2,1,7,...,56,17,16,1.32,2025,LG,LG,NaN,NaN,NaN
4,5,50106,유영찬,LG,2.63,39,2,2,21,1,...,52,14,12,1.32,2025,LG,LG,NaN,NaN,NaN


## 다른 데이터셋 사용
`Hugging Face`의 데이터셋이 결측치가 많고, 주요 지표를 사용할 수 없다는 단점으로 새로운 데이터셋을 구했다.   
출처는 공유자분께서 밝히지 말아달라고 하셔서 기말 과제 보고서에만 작성해서 제출할 예정이다.


In [7]:
# 로컬 파일 업로드
from google.colab import files

upload = files.upload()

Saving kbo_batting_stats_by_season_1982-2025.csv to kbo_batting_stats_by_season_1982-2025 (1).csv
Saving kbo_pitching_stats_by_season_1982-2025.csv to kbo_pitching_stats_by_season_1982-2025 (1).csv


In [8]:
# csv 파일 읽기
import pandas as pd

hitter = pd.read_csv("kbo_batting_stats_by_season_1982-2025.csv", encoding = "utf-8-sig")
pitcher = pd.read_csv("kbo_pitching_stats_by_season_1982-2025.csv", encoding = "utf-8-sig")

hitter.head()
pitcher.head()

,Id,Name,Birthdate,Handedness,School,Draft,Year,Team,Age,Pos.,...,SO,ROE,BK,WP,ERA,RA9,rRA9,FIP,WHIP,WAR
0,1038,박영진,1958년 07월 27일,우투우타,대구상고-성균관대,82 삼성 창단 멤버,1982,삼성,24,P,...,3,NaN,1,0,8.64,9.72,9.72,6.51,2.40,-0.17
1,1038,박영진,1958년 07월 27일,우투우타,대구상고-성균관대,82 삼성 창단 멤버,1984,삼성,26,P,...,0,NaN,0,0,18.00,18.00,18.00,2.43,4.00,-0.06
2,1057,김동철,1960년 06월 05일,우투우타,동산고,82 삼미 창단 멤버,1982,삼미,22,P,...,19,NaN,1,2,7.06,8.13,8.13,6.04,1.90,-1.33
3,1059,김용남,1958년 02월 26일,우투우타,군산초-군산중-군산상고-한양대,82 해태 창단 멤버,1982,해태,24,P,...,85,NaN,0,4,3.09,4.27,4.27,3.17,1.25,3.34
4,1059,김용남,1958년 02월 26일,우투우타,군산초-군산중-군산상고-한양대,82 해태 창단 멤버,1983,해태,25,P,...,80,NaN,0,1,2.83,3.33,3.33,2.56,1.11,4.58


### Github에 업로드 한 2026 raw 데이터 불러오기


In [9]:
golden_glove = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_goldenglove_data_2025.csv")
hitter_2026 = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_hitter_data_2026.csv")
pitcher_2026 = pd.read_csv("https://raw.githubusercontent.com/dohee-jin/data-mining-final-project/main/data/raw/kbo_pitcher_data_2026.csv")

golden_glove.head()
hitter_2026.head()
pitcher_2026.head()

,rank,name,team,ERA,G,W,L,SV,HLB,WPCT,...,NP,AVG,2B,3B,SAC,SF,IBB,WP,BK,year
0,1,후라도,삼성,2.61,12,3,1,0,0,0.750,...,1156,0.259,6,2,4,3,0,4,0,2026
1,2,올러,KIA,2.66,13,7,5,0,0,0.583,...,1211,0.182,5,2,5,1,0,3,0,2026
2,3,류현진,한화,2.84,12,8,2,0,0,0.800,...,1059,0.239,14,1,7,2,1,1,0,2026
3,4,최민석,두산,2.88,12,6,2,0,0,0.750,...,1098,0.215,13,0,3,2,0,3,1,2026
4,5,알칸타라,키움,3.12,12,6,4,0,0,0.600,...,1154,0.259,12,2,1,1,0,2,0,2026


## 데이터 전처리
### 1. Kaggle 데이터와 KBO 2026 데이터 칼럼명 맞추기

#### 기록용어 참고
01. 타자 기록
  - 2B: 2루타
  - 3B: 3루타
  - AB: 타수
  - AO: 뜬공
  - AVG: 타율
  - BB: 볼넷
  - BB/K: 볼넷/삼진
  - CS: 도루실패
  - E: 실책
  - G: 경기
  - GDP: 병살타
  - GO : 땅볼
  - GO/AO : 땅볼/뜬공
  - GPA : (1.8x출루율+장타율)/4
  - GW RBI : 결승타
  - H : 안타
  - HBP(HP) : 사구
  - HR : 홈런
  - IBB(IB) : 고의4구
  - ISOP : 순수장타율
  - MH : 멀티히트
  - OBP : 출루율
  - OPS : 출루율+장타율
  - P/PA : 투구수/타석
  - PA : 타석
  - PH-BA : 대타타율
  - R : 득점
  - RBI : 타점
  - RISP : 득점권타율
  - SAC(SH) : 희생번트
  - SB : 도루
  - SF : 희생플라이
  - SLG : 장타율
  - SO : 삼진
  - TB: 루타
  - XBH : 장타
  - XR : 추정득점

02. 투수기록
  - 2B : 2루타
  - 3B : 3루타
  - AO : 뜬공
  - AVG : 피안타율
  - BABIP : 인플레이타구타율
  - BB : 볼넷
  - BB/9 : 9이닝당 볼넷
  - BK : 보크
  - BSV : 블론세이브
  - CG : 완투
  - ER : 자책점
  - ERA : 평균자책점
  - G : 경기
  - GDP : 병살타
  - GF : 종료
  - GO : 땅볼
  - GO/AO : 땅볼/뜬공
  - GS : 선발
  - H : 피안타
  - HBP(HP) : 사구
  - HLD(HD) : 홀드
  - HR : 홈런
  - IBB(IB) : 고의4구
  - IP : 이닝
  - K/9 : 9이닝당 삼진
  - K/BB : 삼진/볼넷
  - L : 패
  - NP : 투구수
  - OBP : 피출루율
  - OPS : 피출루율+피장타율
  - P/G : 투구수/경기
  - P/IP : 투구수/이닝
  - QS : 퀄리티스타트
  - R : 실점
  - SAC(SH) : 희생번트
  - SF : 희생플라이
  - SHO : 완봉
  - SLG : 피장타율
  - SO : 삼진
  - SV(S) : 세이브
  - SVO : 세이브기회
  - TBF : 타자수
  - TS : 터프세이브
  - W : 승
  - Wgr : 구원승
  - Wgs : 선발승
  - WHIP : 이닝당 출루허용률
  - WP : 폭투
  - WPCT : 승률

####
   

In [10]:
# 각 데이터의 칼럼명 확인
print(hitter.columns)
print(pitcher.columns)
print(hitter_2026.columns)
print(pitcher_2026.columns)
print(golden_glove.columns)


Index(['Id', 'Name', 'Birthdate', 'Handedness', 'School', 'Draft', 'Year',
       'Team', 'Age', 'Pos.', 'G', 'oWAR', 'dWAR', 'PA', 'ePA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'TB', 'RBI', 'SB', 'CS', 'BB', 'HP', 'IB', 'SO',
       'GDP', 'SH', 'SF', 'AVG', 'OBP', 'SLG', 'OPS', 'R/ePA', 'wRC+', 'WAR'],
      dtype='object')
Index(['Id', 'Name', 'Birthdate', 'Handedness', 'School', 'Draft', 'Year',
       'Team', 'Age', 'Pos.', 'G', 'GS', 'GR', 'GF', 'CG', 'SHO', 'W', 'L',
       'S', 'HD', 'IP', 'ER', 'R', 'rRA', 'TBF', 'H', '2B', '3B', 'HR', 'BB',
       'HP', 'IB', 'SO', 'ROE', 'BK', 'WP', 'ERA', 'RA9', 'rRA9', 'FIP',
       'WHIP', 'WAR'],
      dtype='object')
Index(['rank', 'name', 'team', 'position', 'AVG', 'G', 'PA', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'BB', 'IBB', 'HBP', 'SO',
       'GDP', 'SLG', 'OBP', 'OPS', 'MH', 'RISP', 'PH-BA', 'year'],
      dtype='object')
Index(['rank', 'name', 'team', 'ERA', 'G', 'W', 'L', 'SV', 'HLB', 'WPCT', 'IP',


In [11]:
# 타자 데이터 컬럼명 정리(Kaggle 기준으로 정리)

hitter_2026 = hitter_2026.rename(columns={
    "name": "Name",
    "team": "Team",
    "year": "Year",
    "position": "Pos",
    "HBP": "HP",
    "IBB": "IB",
    "SAC": "SH"
})

"""
# 2026 데이터 기준으로 정리
hitter_recent = hitter_recent.rename(columns = {
    "순위": "rank",
    "선수명": "name",
    "팀명": "team",
})

pitcher_recent = pitcher_recent.rename(columns = {
    "순위": "rank",
    "선수명": "name",
    "팀명": "team",
})

# Strip whitespace from name and team columns
hitter_recent["name"] = hitter_recent["name"].str.strip()
hitter_recent["team"] = hitter_recent["team"].str.strip()
pitcher_recent["name"] = pitcher_recent["name"].str.strip()
pitcher_recent["team"] = pitcher_recent["team"].str.strip()
"""

# 투수 데이터 컬럼명 정리(Kaggle 기준으로 정리)
pitcher_2026 = pitcher_2026.rename(columns={
    "name": "Name",
    "team": "Team",
    "year": "Year",
    "HLB": "HD",
    "SV": "S"
})

print(hitter_2026.columns)
print(pitcher_2026.columns)
# print(hitter_recent.columns)
# print(pitcher_recent.columns)

Index(['rank', 'Name', 'Team', 'Pos', 'AVG', 'G', 'PA', 'AB', 'R', 'H', '2B',
       '3B', 'HR', 'TB', 'RBI', 'SH', 'SF', 'BB', 'IB', 'HP', 'SO', 'GDP',
       'SLG', 'OBP', 'OPS', 'MH', 'RISP', 'PH-BA', 'Year'],
      dtype='object')
Index(['rank', 'Name', 'Team', 'ERA', 'G', 'W', 'L', 'S', 'HD', 'WPCT', 'IP',
       'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'CG', 'SHO', 'QS',
       'BSV', 'TBF', 'NP', 'AVG', '2B', '3B', 'SAC', 'SF', 'IBB', 'WP', 'BK',
       'Year'],
      dtype='object')


### 2. 골든글러브 데이터 전처리
#### 2-1. 칼럼명 정리
year, P, C, 1B, 2B, 3B, SS, OF1, OF2, OF3, OF4, DH(기존) --> year, position, player_team(전처리 후)

#### 2-2. 선수명/팀명 분리
현재 데이터에는 `폰세 한화`로 되어있어서 분리가 필요하다.

#### 2-3. 2015-2025 학습 데이터에 `is_goldenglove` 라벨 추가

In [12]:
# 2-1. 칼럼 및 데이터 정리
position_cols = ["P", "C", "1B", "2B", "3B", "SS", "OF1", "OF2", "OF3", "OF4", "DH"]

golden = golden_glove.melt(
    id_vars = "year",
    value_vars = position_cols,
    var_name = "position",
    value_name = "player_name"
)

# 빈 값 제거
golden = golden.dropna(subset=["player_name"])

# OF1..4 -> OF로 통일
golden["position"] = golden["position"].replace({
    "OF1": "OF",
    "OF2": "OF",
    "OF3": "OF",
    "OF4": "OF"
})

# 선수명, 팀이름 분리
golden[["name", 'team']] = golden["player_name"].str.rsplit(" ", n = 1, expand = True)

# Strip whitespace after splitting
golden["name"] = golden["name"].str.strip()
golden["team"] = golden["team"].str.strip()

golden = golden[["year", "position", "name", "team"]]

# Rename columns to match Kaggle data's capitalization for merging/matching later
golden = golden.rename(columns={"year": "Year", "name": "Name", "team": "Team"})

golden.head()

,Year,position,Name,Team
0,2025,P,폰세,한화
1,2024,P,하트,NC
2,2023,P,페디,NC
3,2022,P,안우진,키움
4,2021,P,미란다,두산


In [13]:
# 학습용 데이터에 is_goldenglove 라벨 붙이기

# 원본 데이터 복사해서 사용
hitter_recent = hitter[hitter['Year'] >= 2000].copy()
pitcher_recent = pitcher[pitcher['Year'] >= 2000].copy()

hitter_recent["is_goldenglove"] = hitter_recent.set_index(["Year", "Name", "Team"]).index.isin(
    golden.set_index(["Year", "Name", "Team"]).index
).astype(int)

pitcher_recent["is_goldenglove"] = pitcher_recent.set_index(["Year", "Name", "Team"]).index.isin(
  golden.set_index(["Year", "Name", "Team"]).index
).astype(int)

hitter_recent.head()
pitcher_recent.head()

# is_goldenglove 라벨 확인: 골든글러브 수상자면 1, 아니면 0
hitter_recent[hitter_recent["Name"] == "김도영"].head()


,Id,Name,Birthdate,Handedness,School,Draft,Year,Team,Age,Pos.,...,SH,SF,AVG,OBP,SLG,OPS,R/ePA,wRC+,WAR,is_goldenglove
9528,15056,김도영,2003년 10월 02일,우투우타,광주대성초-광주동성중-광주동성고,22 KIA 1차,2022,KIA,19,3B,...,4,1,0.237,0.312,0.362,0.674,0.104,90.7,1.75,0
9529,15056,김도영,2003년 10월 02일,우투우타,광주대성초-광주동성중-광주동성고,22 KIA 1차,2023,KIA,20,3B,...,2,4,0.303,0.371,0.453,0.824,0.156,134.4,3.76,0
9530,15056,김도영,2003년 10월 02일,우투우타,광주대성초-광주동성중-광주동성고,22 KIA 1차,2024,KIA,21,3B,...,1,7,0.347,0.420,0.647,1.067,0.232,172.5,8.59,1
9531,15056,김도영,2003년 10월 02일,우투우타,광주대성초-광주동성중-광주동성고,22 KIA 1차,2025,KIA,22,3B,...,0,2,0.309,0.361,0.582,0.943,0.181,152.4,1.33,0


In [14]:
# 라벨이 잘 붙었는지 확인

print(hitter_recent["is_goldenglove"].value_counts())
print(pitcher_recent["is_goldenglove"].value_counts())

print(hitter_recent.groupby("Year")["is_goldenglove"].sum())
print(pitcher_recent.groupby("Year")["is_goldenglove"].sum())

is_goldenglove
0    6143
1     225
Name: count, dtype: int64
is_goldenglove
0    5726
1      26
Name: count, dtype: int64
Year
2000    9
2001    9
2002    9
2003    9
2004    9
2005    9
2006    9
2007    9
2008    8
2009    8
2010    9
2011    9
2012    9
2013    9
2014    9
2015    7
2016    8
2017    8
2018    8
2019    9
2020    9
2021    9
2022    8
2023    9
2024    9
2025    8
Name: is_goldenglove, dtype: int64
Year
2000    1
2001    1
2002    1
2003    1
2004    1
2005    1
2006    1
2007    1
2008    1
2009    1
2010    1
2011    1
2012    1
2013    1
2014    1
2015    1
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    1
2023    1
2024    1
2025    1
Name: is_goldenglove, dtype: int64


In [42]:
team_corrections = {
    (2013, "정근우"): "SK",
    (2015, "유한준"): "넥센",
    # (2015, "박석민"): "삼성",  # 사용자 피드백에 따라 제거: golden_glove 원본에 이미 박석민 NC로 되어 있음
    (2017, "강민호"): "롯데",
    (2016, "최형우"): "삼성",
    (2022, "양의지"): "NC",
    (2025, "최형우"): "KIA"
}

for (year, name), team in team_corrections.items():
    golden.loc[
        (golden["Year"] == year) & (golden["Name"] == name),
        "Team"
    ] = team

# 2018 이정후 선수의 Name, Team 분리 안되는 문제 해결
golden.loc[
    (golden["Year"] == 2018) &
    (golden["Team"].str.contains("이정후", na=False)),
    ["Name", "Team"]
] = ["이정후", "넥센"]

# Print golden for 2013 and 2015 to verify after corrections
print("Golden dataframe for 2013 after corrections:")
display(golden[golden["Year"] == 2013])

print("\nGolden dataframe for 2015 after corrections:")
display(golden[golden["Year"] == 2015])

Golden dataframe for 2013 after corrections:


,Year,position,Name,Team
12,2013,P,손승락,넥센
38,2013,C,강민호,롯데
64,2013,1B,박병호,넥센
90,2013,2B,정근우,SK
116,2013,3B,최정,SK
142,2013,SS,강정호,넥센
168,2013,OF,박용택,LG
194,2013,OF,손아섭,롯데
220,2013,OF,최형우,삼성
272,2013,DH,이병규,LG



Golden dataframe for 2015 after corrections:


,Year,position,Name,Team
10,2015,P,해커,NC
36,2015,C,양의지,두산
62,2015,1B,테임즈,NC
88,2015,2B,나바로,삼성
114,2015,3B,박석민,삼성
140,2015,SS,김재호,두산
166,2015,OF,김현수,두산
192,2015,OF,나성범,NC
218,2015,OF,유한준,넥센
270,2015,DH,이승엽,삼성


In [34]:
# golden_hitter_recent 생성 후 매칭 확인
hitter_positions = ["C", "1B", "2B", "3B", "SS", "OF", "DH"]

golden_hitter_recent = golden[
    (golden["position"].isin(hitter_positions)) &
    (golden["Year"] >= 2016) &
    (golden["Year"] <= 2025)
].copy()

hitter_keys = set(zip(hitter_recent["Year"], hitter_recent["Name"], hitter_recent["Team"]))

golden_hitter_recent["matched"] = golden_hitter_recent.apply(
    lambda row: (row["Year"], row["Name"], row["Team"]) in hitter_keys,
    axis=1
)

failed_hitter = golden_hitter_recent[golden_hitter_recent["matched"] == False]

print("매칭 실패 인원:", len(failed_hitter))
display(failed_hitter)

print(golden["Year"].dtype)
print(hitter_recent["Year"].dtype)

print(golden["Name"].dtype)
print(hitter_recent["Name"].dtype)

print(golden["Team"].dtype)
print(hitter_recent["Team"].dtype)

매칭 실패 인원: 0


,Year,position,Name,Team,matched


Int64
Int64
object
object
object
object


In [35]:
# 라벨 여부 한번 더 확인

pd.set_option("display.max_rows", None)      # 행 전체 출력
pd.set_option("display.max_columns", None)   # 열 전체 출력
pd.set_option("display.width", None)         # 줄바꿈 제한 완화
pd.set_option("display.max_colwidth", None)  # 셀 내용 전체 출력

hitter_recent[hitter_recent["is_goldenglove"] == 1]
pitcher_recent[pitcher_recent["is_goldenglove"] == 1]

,Id,Name,Birthdate,Handedness,School,Draft,Year,Team,Age,Pos.,G,GS,GR,GF,CG,SHO,W,L,S,HD,IP,ER,R,rRA,TBF,H,2B,3B,HR,BB,HP,IB,SO,ROE,BK,WP,ERA,RA9,rRA9,FIP,WHIP,WAR,is_goldenglove
1457,1716,정민태,1970년 03월 01일,우투우타,숭의초-동산중-동산고-한양대,92 태평양 1차,2003,현대,33,P,29,29,0,1,1,0,17,2,0,0,177.0,65,76,76.00,741,179,NaN,NaN,17,42,5,1,122,NaN,0,3,3.31,3.86,3.86,3.40,1.25,4.60,1
1980,2013,임선동,1973년 08월 04일,우투우타,사당초-휘문중-휘문고-연세대,92 LG 1차,2000,현대,27,P,29,29,0,1,1,1,18,4,0,0,195.1,73,75,75.00,802,177,NaN,NaN,13,61,7,0,174,NaN,0,8,3.36,3.46,3.46,3.27,1.22,5.77,1
2230,2254,리오스,1972년 11월 11일,우투우타,-,02 KIA 자유선발,2007,두산,35,P,33,33,0,6,6,4,22,5,0,0,234.2,54,69,69.00,947,191,NaN,NaN,8,58,16,2,147,NaN,0,5,2.07,2.65,2.65,2.99,1.06,8.19,1
2429,10048,로페즈,1975년 04월 21일,우투우타,-,09 KIA 자유선발,2009,KIA,34,P,29,26,3,7,4,0,14,5,0,0,190.1,66,83,83.00,802,200,NaN,NaN,6,41,10,0,129,NaN,0,3,3.12,3.92,3.92,3.16,1.27,4.83,1
2498,10058,양현종,1988년 03월 01일,좌투좌타,광주학강초-광주동성중-광주동성고,07 KIA 2차 1라운드 1순위,2017,KIA,29,P,31,31,0,1,1,0,20,6,0,0,193.1,74,88,85.25,808,209,32.0,2.0,17,45,0,0,158,15.0,0,14,3.44,4.10,3.97,3.93,1.31,5.45,1
2529,10061,윤석민,1986년 07월 24일,우투우타,구리초-구리인창중-야탑고,05 KIA 2차 1라운드 6순위,2011,KIA,25,P,27,25,2,5,3,3,17,5,1,0,172.1,47,53,53.00,679,137,NaN,NaN,10,44,6,0,178,NaN,1,2,2.45,2.77,2.77,2.63,1.05,5.94,1
2697,10126,김광현,1988년 07월 22일,좌투좌타,안산-덕성초-안산중앙중-안산공고,07 SK 1차 1순위,2008,SK,20,P,27,27,0,1,1,1,16,4,0,0,162.0,43,50,50.00,659,127,NaN,NaN,9,63,2,1,150,NaN,0,3,2.39,2.78,2.78,3.09,1.17,5.69,1
3408,10284,손민한,1975년 01월 02일,우투우타,대연초-대천중-부산고-고려대,97 롯데 1차,2005,롯데,30,P,28,26,2,3,1,0,18,7,1,0,168.1,46,54,54.00,678,149,NaN,NaN,9,38,2,2,105,NaN,0,4,2.46,2.89,2.89,3.22,1.11,5.70,1
3676,10362,배영수,1981년 05월 04일,우투우타,칠성초-경복중-경북고,00 삼성 1차,2004,삼성,23,P,35,27,8,5,4,2,17,2,0,0,189.2,55,65,65.00,792,163,NaN,NaN,6,74,11,1,144,NaN,1,13,2.61,3.08,3.08,3.23,1.25,6.18,1
3775,10375,장원삼,1983년 06월 09일,좌투좌타,사파초-신월중-용마고-경성대,02 현대 2차 11라운드 89순위,2012,삼성,29,P,27,25,2,0,0,0,17,6,0,1,157.0,62,64,64.00,649,143,NaN,NaN,9,38,9,1,127,NaN,0,2,3.55,3.67,3.67,3.05,1.15,3.31,1


In [46]:
# is_goldenglove 라벨 재생성

# 2018 이정후 매칭 보정
# golden에는 2018 이정후가 있는데 매칭이 안 되는 문제를 수동 보정

golden.loc[
    (golden["Year"] == 2018) &
    (golden["Name"].astype(str).str.contains("이정후", na=False)),
    ["Name", "Team"]
] = ["이정후", "넥센"]

# 1. 타입/문자열 정리
golden["Year"] = pd.to_numeric(golden["Year"], errors="coerce").astype("Int64")
hitter_recent["Year"] = pd.to_numeric(hitter_recent["Year"], errors="coerce").astype("Int64")
pitcher_recent["Year"] = pd.to_numeric(pitcher_recent["Year"], errors="coerce").astype("Int64")

for col in ["Name", "Team"]:
    golden[col] = (
        golden[col]
        .astype(str)
        .str.replace("\xa0", " ", regex=False)
        .str.replace("\u200b", "", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

for col in ["Name", "Team"]: # Changed: Changed 'name', 'team' to 'Name', 'Team'
    hitter_recent[col] = (
        hitter_recent[col]
        .astype(str)
        .str.replace("\xa0", " ", regex=False)
        .str.replace("\u200b", "", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    pitcher_recent[col] = (
        pitcher_recent[col]
        .astype(str)
        .str.replace("\xa0", " ", regex=False)
        .str.replace("\u200b", "", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# 2. 타자/투수 골든글러브 정답 목록 분리
hitter_positions = ["C", "1B", "2B", "3B", "SS", "OF", "DH"]

golden_hitter_recent = golden[
    (golden["position"].isin(hitter_positions)) &
    (golden["Year"] >= 2000) &
    (golden["Year"] <= 2025)
].copy()

golden_pitcher_recent = golden[
    (golden["position"] == "P") &
    (golden["Year"] >= 2000) &
    (golden["Year"] <= 2025)
].copy()

# 3. 중복 제거
golden_hitter_recent = golden_hitter_recent.drop_duplicates(
    subset=["Year", "Name", "Team", "position"]
)

golden_pitcher_recent = golden_pitcher_recent.drop_duplicates(
    subset=["Year", "Name", "Team"]
)

hitter_recent = hitter_recent.drop_duplicates(
    subset=["Year", "Name", "Team"],
    keep="first"
).copy()

pitcher_recent = pitcher_recent.drop_duplicates(
    subset=["Year", "Name", "Team"],
    keep="first"
).copy()

# 4. 라벨 다시 생성
hitter_recent["is_goldenglove"] = hitter_recent.set_index(
    ["Year", "Name", "Team"]
).index.isin(
    golden_hitter_recent.set_index(["Year", "Name", "Team"]).index
).astype(int)

pitcher_recent["is_goldenglove"] = pitcher_recent.set_index(
    ["Year", "Name", "Team"]
).index.isin(
    golden_pitcher_recent.set_index(["Year", "Name", "Team"]).index
).astype(int)

# 5. 결과 확인
print("타자 연도별 골든글러브 라벨 수")
print(hitter_recent.groupby("Year")["is_goldenglove"].sum()) # Changed 'year' to 'Year'

print("\n투수 연도별 골든글러브 라벨 수")
print(pitcher_recent.groupby("Year")["is_goldenglove"].sum()) # Changed 'year' to 'Year'

print("\n타자 라벨 분포")
print(hitter_recent["is_goldenglove"].value_counts())

print("\n투수 라벨 분포")
print(pitcher_recent["is_goldenglove"].value_counts())

타자 연도별 골든글러브 라벨 수
Year
2000    9
2001    9
2002    9
2003    9
2004    9
2005    9
2006    9
2007    9
2008    8
2009    8
2010    9
2011    9
2012    9
2013    9
2014    9
2015    9
2016    9
2017    9
2018    9
2019    9
2020    9
2021    9
2022    9
2023    9
2024    9
2025    9
Name: is_goldenglove, dtype: int64

투수 연도별 골든글러브 라벨 수
Year
2000    1
2001    1
2002    1
2003    1
2004    1
2005    1
2006    1
2007    1
2008    1
2009    1
2010    1
2011    1
2012    1
2013    1
2014    1
2015    1
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    1
2023    1
2024    1
2025    1
Name: is_goldenglove, dtype: int64

타자 라벨 분포
is_goldenglove
0    6125
1     232
Name: count, dtype: int64

투수 라벨 분포
is_goldenglove
0    5713
1      26
Name: count, dtype: int64


In [48]:
# 2013년과 2015년 골든글러브 수상자 목록 (golden DataFrame)
golden_2013 = golden[golden['Year'] == 2013]
golden_2015 = golden[golden['Year'] == 2015]

print("\n--- golden DataFrame (2013) ---")
display(golden_2013)
print("--- golden DataFrame (2015) ---")
display(golden_2015)

# hitter_recent, pitcher_recent에 is_goldenglove 라벨이 1인 선수들 (2013년과 2015년)
hitter_recent_goldenglove_2013 = hitter_recent[(hitter_recent['Year'] == 2013) & (hitter_recent['is_goldenglove'] == 1)]
hitter_recent_goldenglove_2015 = hitter_recent[(hitter_recent['Year'] == 2015) & (hitter_recent['is_goldenglove'] == 1)]

pitcher_recent_goldenglove_2013 = pitcher_recent[(pitcher_recent['Year'] == 2013) & (pitcher_recent['is_goldenglove'] == 1)]
pitcher_recent_goldenglove_2015 = pitcher_recent[(pitcher_recent['Year'] == 2015) & (pitcher_recent['is_goldenglove'] == 1)]


print("\n--- recent with is_goldenglove=1 (2013) ---")
display(hitter_recent_goldenglove_2013[['Year', 'Name', 'Team', 'is_goldenglove']])
display(pitcher_recent_goldenglove_2013[['Year', 'Name', 'Team', 'is_goldenglove']])
print("--- recent with is_goldenglove=1 (2015) ---")
display(hitter_recent_goldenglove_2015[['Year', 'Name', 'Team', 'is_goldenglove']])
display(pitcher_recent_goldenglove_2013[['Year', 'Name', 'Team', 'is_goldenglove']])

# golden에 있지만 hitter_recent와 pitcher_recent에서 is_goldenglove=1이 아닌 선수들 찾기
# 2013년
golden_keys_2013 = set(zip(golden_2013['Name'], golden_2013['Team']))
recent_keys_2013 = (
    set(zip(hitter_recent_goldenglove_2013['Name'], hitter_recent_goldenglove_2013['Team']))
    |
    set(zip(pitcher_recent_goldenglove_2013['Name'], pitcher_recent_goldenglove_2013['Team']))
)

missing_in_2013 = [player for player in golden_keys_2013 if player not in recent_keys_2013]

print("\n--- 2013년 golden에는 있으나 hitter_recent, pitcher_recent에서 라벨링되지 않은 선수 (Name, Team) ---")
if missing_in_2013:
    for name, team in missing_in_2013:
        print(f"이름: {name}, 팀: {team}")
        # hitter_recent에서 이 선수의 정보를 찾아 어떤 문제가 있는지 확인 (예: Name, Team 불일치)
        display(hitter_recent[(hitter_recent['Year'] == 2013)
                                & (hitter_recent['Name'] == name)
                                & (pitcher_recent['Name'] == name)
                                & (pitcher_recent['Name'] == 2013)
                              ])
else:
    print("2013년에는 매칭되지 않는 선수가 없습니다. 다른 문제가 있을 수 있습니다.")

# 2015년
golden_keys_2015 = set(zip(golden_2015['Name'], golden_2015['Team']))
recent_keys_2015 = (
    set(zip(hitter_recent_goldenglove_2015['Name'], hitter_recent_goldenglove_2015['Team']))
    |
    set(zip(pitcher_recent_goldenglove_2015['Name'], pitcher_recent_goldenglove_2015['Team']))
)

missing_in_2015 = [player for player in golden_keys_2015 if player not in recent_keys_2015]

print("\n--- 2015년 golden에는 있으나 hitter_recent, pitcher_recent에서 라벨링되지 않은 선수 (Name, Team) ---")
if missing_in_2015:
    for name, team in missing_in_2015:
        print(f"이름: {name}, 팀: {team}")
        # hitter_recent에서 이 선수의 정보를 찾아 어떤 문제가 있는지 확인
        display(hitter_recent[(hitter_recent['Year'] == 2015)
                                & (hitter_recent['Name'] == name)
                                & (pitcher_recent['Name'] == name)
                                & (pitcher_recent['Name'] == 2015)
                              ])
else:
    print("2015년에는 매칭되지 않는 선수가 없습니다. 다른 문제가 있을 수 있습니다.")


--- golden DataFrame (2013) ---


,Year,position,Name,Team
12,2013,P,손승락,넥센
38,2013,C,강민호,롯데
64,2013,1B,박병호,넥센
90,2013,2B,정근우,SK
116,2013,3B,최정,SK
142,2013,SS,강정호,넥센
168,2013,OF,박용택,LG
194,2013,OF,손아섭,롯데
220,2013,OF,최형우,삼성
272,2013,DH,이병규,LG


--- golden DataFrame (2015) ---


,Year,position,Name,Team
10,2015,P,해커,NC
36,2015,C,양의지,두산
62,2015,1B,테임즈,NC
88,2015,2B,나바로,삼성
114,2015,3B,박석민,삼성
140,2015,SS,김재호,두산
166,2015,OF,김현수,두산
192,2015,OF,나성범,NC
218,2015,OF,유한준,넥센
270,2015,DH,이승엽,삼성



--- recent with is_goldenglove=1 (2013) ---


...

,Year,Name,Team,is_goldenglove
4029,2013,손승락,넥센,1


--- recent with is_goldenglove=1 (2015) ---


,Year,Name,Team,is_goldenglove
4487,2015,양의지,두산,1
4607,2015,김재호,두산,1
4732,2015,김현수,두산,1
5357,2015,박석민,삼성,1
6701,2015,유한준,넥센,1
7246,2015,이승엽,삼성,1
7594,2015,나성범,NC,1
7660,2015,나바로,삼성,1
7800,2015,테임즈,NC,1


,Year,Name,Team,is_goldenglove
4029,2013,손승락,넥센,1



--- 2013년 golden에는 있으나 hitter_recent, pitcher_recent에서 라벨링되지 않은 선수 (Name, Team) ---
2013년에는 매칭되지 않는 선수가 없습니다. 다른 문제가 있을 수 있습니다.

--- 2015년 golden에는 있으나 hitter_recent, pitcher_recent에서 라벨링되지 않은 선수 (Name, Team) ---
2015년에는 매칭되지 않는 선수가 없습니다. 다른 문제가 있을 수 있습니다.


In [50]:
# 타자 결측치 비율 확인(NaN)
hitter_missing = hitter_recent.isna().mean().sort_values(ascending=False)
print("타자 결측치 비율")
display(hitter_missing[hitter_missing > 0])


타자 결측치 비율


,0
AVG,0.001258
SLG,0.001258
OPS,0.001258
Draft,0.000787
Handedness,0.000157


In [51]:
# 투수 결측치 비율 확인(NaN)
pitcher_missing = pitcher_recent.isna().mean().sort_values(ascending=False)
print("투수 결측치 비율")
display(pitcher_missing[pitcher_missing > 0])

투수 결측치 비율


,0
ROE,0.408608
3B,0.408608
2B,0.408608
RA9,0.002265
ERA,0.002265
WAR,0.001917
rRA9,0.001917
Handedness,0.000697
WHIP,0.000174


In [80]:
import numpy as np

# 학습 데이터의 불필요한 공통 칼럼 제거

# 공통 삭제 칼럼
extra_drop_cols = ["rank", "School", "Draft", "Id"]

# 원본 copy
hitter_recent_clean = hitter_recent.copy()
pitcher_recent_clean = pitcher_recent.copy()

# 불필요한 칼럼 제거
hitter_cols_to_drop = [col for col in extra_drop_cols if col in hitter_recent_clean.columns]
if hitter_cols_to_drop:
    hitter_recent_clean = hitter_recent_clean.drop(columns=hitter_cols_to_drop)

pitcher_cols_to_drop = [col for col in extra_drop_cols if col in pitcher_recent_clean.columns]
if pitcher_cols_to_drop:
    pitcher_recent_clean = pitcher_recent_clean.drop(columns=pitcher_cols_to_drop)

hitter_recent_clean.head()
pitcher_recent_clean.head()

# 2B, 3B, ROE의 값이 NaN인것을 확인 할 수 있음


,Name,Birthdate,Handedness,Year,Team,Age,Pos.,G,GS,GR,GF,CG,SHO,W,L,S,HD,IP,ER,R,rRA,TBF,H,2B,3B,HR,BB,HP,IB,SO,ROE,BK,WP,ERA,RA9,rRA9,FIP,WHIP,WAR,is_goldenglove
434,김용수,1960년 05월 02일,우투우타,2000,LG,40,P,32,19,13,10,0,0,6,4,4,1,127.0,74,83,83.0,556,142,NaN,NaN,10,42,9,2,85,NaN,0,4,5.24,5.88,5.88,4.09,1.45,0.58,0
514,김정수,1962년 07월 24일,좌투우타,2000,SK,38,P,50,0,50,2,0,0,1,4,0,3,31.0,26,26,26.0,141,40,NaN,NaN,5,14,3,0,23,NaN,0,1,7.55,7.55,7.55,5.39,1.74,-0.14,0
515,김정수,1962년 07월 24일,좌투우타,2001,한화,39,P,52,0,52,12,0,0,2,2,1,10,43.2,18,24,24.0,192,36,NaN,NaN,6,19,5,3,45,NaN,1,4,3.71,4.95,4.95,4.09,1.26,0.82,0
516,김정수,1962년 07월 24일,좌투우타,2002,한화,40,P,47,0,47,6,0,0,1,2,1,12,25.0,9,10,10.0,104,21,NaN,NaN,3,8,4,2,29,NaN,0,3,3.24,3.60,3.60,3.25,1.16,0.64,0
517,김정수,1962년 07월 24일,좌투우타,2003,한화,41,P,3,0,3,0,0,0,0,0,0,0,1.0,0,0,0.0,5,1,NaN,NaN,0,1,0,0,0,NaN,0,1,0.00,0.00,0.00,6.28,2.00,0.08,0


In [81]:
def convert_ip_to_float(ip_str):
    if pd.isna(ip_str) or ip_str == '-':
        return np.nan
    if isinstance(ip_str, (int, float)):
        return float(ip_str)
    parts = str(ip_str).split(' ')
    if len(parts) == 1:
        return float(parts[0])
    elif len(parts) == 2 and '/' in parts[1]:
        integer_part = float(parts[0])
        fraction_parts = parts[1].split('/')
        numerator = float(fraction_parts[0])
        denominator = float(fraction_parts[1])
        return integer_part + (numerator / denominator)
    return np.nan

In [82]:
# 공통 칼럼 삭제 후 데이터프레임의 결측치 비율 확인
print("\n--- hitter_recent Missing Values ---")
print(hitter_recent_clean.isna().mean().sort_values(ascending=False).head(30))

print("\n--- pitcher_recent Missing Values ---")
print(pitcher_recent_clean.isna().mean().sort_values(ascending=False).head(30))


--- hitter_recent Missing Values ---
OPS           0.001258
AVG           0.001258
SLG           0.001258
Handedness    0.000157
Name          0.000000
Team          0.000000
Year          0.000000
Birthdate     0.000000
Age           0.000000
dWAR          0.000000
Pos.          0.000000
G             0.000000
oWAR          0.000000
R             0.000000
H             0.000000
2B            0.000000
3B            0.000000
HR            0.000000
PA            0.000000
ePA           0.000000
AB            0.000000
SB            0.000000
RBI           0.000000
TB            0.000000
CS            0.000000
SO            0.000000
BB            0.000000
HP            0.000000
IB            0.000000
SF            0.000000
dtype: float64

--- pitcher_recent Missing Values ---
3B            0.408608
ROE           0.408608
2B            0.408608
RA9           0.002265
ERA           0.002265
WAR           0.001917
rRA9          0.001917
Handedness    0.000697
WHIP          0.000174
G          

## 모델 학습
```
1. 지도학습 모델
   - 로지스틱 회귀
   - 의사결정트리
   - 퍼셉트론
   → is_goldenglove = 1/0 예측

2. 비지도학습 분석
   - K-Means 군집화
   → 선수 기록 특성에 따라 선수 유형 분류
```
군집화 분석은 골든글러브 수상 여부를 직접 예측하기 위한 모델이 아니라, 선수들의 기록 패턴을 기준으로 유사한 선수 유형을 파악하기 위한 보조 분석으로 활용하였다. 이를 통해 예측 확률이 높은 선수들이 어떤 기록 특성을 가진 군집에 속하는지 함께 해석하였다.   

### 1. 타자 모델 학습 결과
모델 학습후 Accuracy를 확인했을 때
| 모델      | Accuracy | class 1 Precision | class 1 Recall | 해석                 |
| ------- | -------: | ----------------: | -------------: | ------------------ |
| 로지스틱 회귀 |    0.927 |              0.21 |           0.72 | 후보 예측에 균형적         |
| 의사결정트리  |    0.939 |              0.22 |           0.61 | 설명력 좋음, 정확도 높음     |
| 퍼셉트론    |    0.905 |              0.20 |           1.00 | 수상자를 놓치지 않지만 과다 예측 |

골든글러브 수상자는 전체 선수 중 매우 적기 때문에 데이터 불균형이 크다. 따라서 단순 정확도보다 수상자 클래스(1)에 대한 recall과 f1-score를 함께 고려하였다. 의사결정트리는 가장 높은 정확도를 보였으나, 로지스틱 회귀는 수상자 클래스에 대해 비교적 높은 recall을 보여 후보군 예측에 적합하였다. 퍼셉트론은 수상자 recall이 1.00으로 가장 높았지만 precision이 낮아 수상 가능성이 있는 선수를 넓게 잡는 경향을 보였다.   

타자 골든글러브 예측 모델 평가 결과, 세 모델 모두 90% 이상의 정확도를 보였다. 그러나 골든글러브 수상자는 전체 선수 중 소수이므로 단순 정확도만으로 모델 성능을 판단하기에는 한계가 있다. 수상자 클래스(1)를 기준으로 보면 로지스틱 회귀는 recall 0.72, f1-score 0.33을 보였고, 의사결정트리는 recall 0.61, f1-score 0.33을 보였다. 퍼셉트론은 recall 1.00으로 실제 수상자를 모두 포착했지만 precision이 0.20으로 낮아 수상 후보를 과도하게 넓게 예측하는 경향을 보였다. 따라서 본 분석에서는 예측 확률 해석이 가능하고 후보군 예측에 비교적 적합한 로지스틱 회귀를 중심으로 2026년 골든글러브 후보를 예측하기로 결정했다.

### 2. 투수 모델 학습 결과
| 모델      | Accuracy | class 1 Precision | class 1 Recall | 해석                      |
| ------- | -------: | ----------------: | -------------: | ----------------------- |
| 로지스틱 회귀 |    0.996 |              0.00 |           0.00 | 실제 수상자를 잡지 못함           |
| 의사결정트리  |    0.996 |              0.50 |           0.50 | 테스트셋 기준 가장 양호하지만 표본이 작음 |
| 퍼셉트론    |    0.842 |              0.02 |           1.00 | 수상자를 모두 잡지만 과다 예측       |

투수 부문은 학습 기간 내 골든글러브 수상자가 10명뿐이므로, 모델이 수상자 클래스의 패턴을 충분히 학습하기 어려웠다. 따라서 투수 예측 결과는 최종 수상자를 단정하기보다, 현재 성적 기준으로 수상 가능성이 높은 후보군을 정렬하는 방식으로 해석하였다.   

투수 골든글러브 수상자는 연도별 1명으로 타자 부문보다 클래스(1)의 수가 매우 적다. 이로 인해 train/test 분할 시 테스트 데이터에 포함되는 실제 수상자 수가 적어 precision, recall, f1-score가 불안정하게 나타났다. 따라서 투수 모델의 평가지표는 참고용으로 해석하고, 최종 예측에서는 로지스틱 회귀의 예측 확률을 기준으로 후보군을 순위화하여 결정하기로 했다.   

투수의 경우 분류 정확도 보단 확률 순위로 `2026년 6월 14일 기준 기록상 수상 가능성이 높은 투수 순위`를 확인하는 것이 맞는 방향으로 판단했다.

In [119]:
# 1. feature 만들기

# 타자 feature
hitter_features = [
    'AVG', 'G', 'PA', 'R', 'H', 'HR', 'RBI',
    'BB', 'SO', 'SLG', 'OBP', 'OPS', 'RISP'
]

# 실제 존재하는 칼럼만 사용
hitter_features = [col for col in hitter_features
                   if col in hitter_recent_clean.columns]

# 투수 feature
pitcher_features = [
    'ERA', 'G', 'W', 'S', 'HD', 'WPCT', 'IP',
    'SO', 'WHIP', 'QS'
]

pitcher_features = [col for col in pitcher_features
                    if col in pitcher_recent_clean.columns]

# 'IP' 컬럼 전처리 (문자열 fractional innings to float)
pitcher_recent_clean['IP'] = pitcher_recent_clean['IP'].apply(convert_ip_to_float)

# 수치형 변환
for col in hitter_features:
    hitter_recent_clean[col] = pd.to_numeric(hitter_recent_clean[col], errors="coerce")

for col in pitcher_features:
    pitcher_recent_clean[col] = pd.to_numeric(pitcher_recent_clean[col], errors="coerce")

# 남은 결측치 중앙값 대체
for col in hitter_features: # Changed: Added this loop
    median_value = hitter_recent_clean[col].median()
    hitter_recent_clean[col] = hitter_recent_clean[col].fillna(median_value)

for col in pitcher_features:
    median_value = pitcher_recent_clean[col].median()
    pitcher_recent_clean[col] = pitcher_recent_clean[col].fillna(median_value)


print("\n최종 타자 feature:")
print(hitter_features)

print("\n최종 투수 feature:")
print(pitcher_features)

print("\n타자 feature 결측치 확인:")
print(hitter_recent_clean[hitter_features].isna().sum())

print("\n투수 feature 결측치 확인:")
print(pitcher_recent_clean[pitcher_features].isna().sum())

print("\n타자 데이터 타입:")
print(hitter_recent_clean[hitter_features].dtypes)

print("\n투수 데이터 타입:")
print(pitcher_recent_clean[pitcher_features].dtypes)

print("\n전처리 완료")
print("타자 데이터 크기:", hitter_recent_clean.shape)
print("투수 데이터 크기:", pitcher_recent_clean.shape)



최종 타자 feature:
['AVG', 'G', 'PA', 'R', 'H', 'HR', 'RBI', 'BB', 'SO', 'SLG', 'OBP', 'OPS']

최종 투수 feature:
['ERA', 'G', 'W', 'S', 'HD', 'IP', 'SO', 'WHIP']

타자 feature 결측치 확인:
AVG    0
G      0
PA     0
R      0
H      0
HR     0
RBI    0
BB     0
SO     0
SLG    0
OBP    0
OPS    0
dtype: int64

투수 feature 결측치 확인:
ERA     0
G       0
W       0
S       0
HD      0
IP      0
SO      0
WHIP    0
dtype: int64

타자 데이터 타입:
AVG    float64
G        int64
PA       int64
R        int64
H        int64
HR       int64
RBI      int64
BB       int64
SO       int64
SLG    float64
OBP    float64
OPS    float64
dtype: object

투수 데이터 타입:
ERA     float64
G         int64
W         int64
S         int64
HD        int64
IP      float64
SO        int64
WHIP    float64
dtype: object

전처리 완료
타자 데이터 크기: (6357, 37)
투수 데이터 크기: (5739, 40)


In [120]:
# 학습 데이터 분리
x_hitter = hitter_recent_clean[hitter_features]
y_hitter = hitter_recent_clean["is_goldenglove"]

x_pitcher = pitcher_recent_clean[pitcher_features]
y_pitcher = pitcher_recent_clean["is_goldenglove"]


In [121]:
# 모델 학습 사용 모델 및 의존성 설정
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

In [122]:
import numpy as np

# 타자 모델 학습
X_train, X_test, y_train, y_test = train_test_split(
    x_hitter,
    y_hitter,
    test_size = 0.2,
    random_state = 42,
    stratify = y_hitter
)

hitter_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter = 1000, class_weight = "balanced"))
    ]),
    "Decision Tree" : Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier (
            max_depth = 4,
            random_state = 42,
            class_weight = "balanced"
        ))
    ]),
    "Perceptron": Pipeline([
      ("imputer", SimpleImputer(strategy="median")),
      ("scaler", StandardScaler()),
      ("model", Perceptron(
          max_iter = 1000,
          random_state = 42,
          class_weight = "balanced"
      ))
    ])
}

for name, model in hitter_models.items():
  model.fit(X_train, y_train)
  pred = model.predict(X_test)

  print("===============")
  print(name)
  print("Accuracy: ", accuracy_score(y_test, pred))
  print(classification_report(y_test, pred))

Logistic Regression
Accuracy:  0.8820754716981132
              precision    recall  f1-score   support

           0       1.00      0.88      0.94      1226
           1       0.22      0.91      0.36        46

    accuracy                           0.88      1272
   macro avg       0.61      0.90      0.65      1272
weighted avg       0.97      0.88      0.91      1272

Decision Tree
Accuracy:  0.8663522012578616
              precision    recall  f1-score   support

           0       1.00      0.87      0.93      1226
           1       0.20      0.89      0.33        46

    accuracy                           0.87      1272
   macro avg       0.60      0.88      0.63      1272
weighted avg       0.97      0.87      0.90      1272

Perceptron
Accuracy:  0.8482704402515723
              precision    recall  f1-score   support

           0       1.00      0.85      0.91      1226
           1       0.18      0.93      0.31        46

    accuracy                           0.85    

In [123]:
# 투수 모델 학습
X_train_pitcher, X_test_pitcher, y_train_pitcher, y_test_pitcher = train_test_split(
    x_pitcher,
    y_pitcher,
    test_size = 0.2,
    random_state = 42,
    stratify = y_pitcher
)

pitcher_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter = 1000, class_weight = "balanced"))
    ]),
    "Decision Tree" : Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier (
            max_depth = 4,
            random_state = 42,
            class_weight = "balanced"
        ))
    ]),
    "Perceptron": Pipeline([
      ("imputer", SimpleImputer(strategy="median")),
      ("scaler", StandardScaler()),
      ("model", Perceptron(
          max_iter = 1000,
          random_state = 42,
          class_weight = "balanced"
      ))
    ])
}

for name, model in pitcher_models.items():
  model.fit(X_train_pitcher, y_train_pitcher)
  pred = model.predict(X_test_pitcher)

  print("===============")
  print(name)
  print("Accuracy: ", accuracy_score(y_test_pitcher, pred))
  print(classification_report(y_test_pitcher, pred))

Logistic Regression
Accuracy:  0.975609756097561
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1143
           1       0.15      1.00      0.26         5

    accuracy                           0.98      1148
   macro avg       0.58      0.99      0.63      1148
weighted avg       1.00      0.98      0.98      1148

Decision Tree
Accuracy:  0.9869337979094077
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1143
           1       0.22      0.80      0.35         5

    accuracy                           0.99      1148
   macro avg       0.61      0.89      0.67      1148
weighted avg       1.00      0.99      0.99      1148

Perceptron
Accuracy:  0.8893728222996515
              precision    recall  f1-score   support

           0       1.00      0.89      0.94      1143
           1       0.04      1.00      0.07         5

    accuracy                           0.89     

In [124]:
# 타자 최종 모델: 로지스틱 회귀
# 투수 최종 모델: 의사결정트리
hitter_final_model = hitter_models["Logistic Regression"]
hitter_final_model.fit(x_hitter, y_hitter)

pitcher_final_model = pitcher_models["Logistic Regression"]
pitcher_final_model.fit(x_pitcher, y_pitcher)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

## 2026 골든글러브 예측
데이터는 2026-06-14일 데이터를 기준으로 확인한다.   

본 분석에서는 골든글러브 수상 여부를 0과 1로 구분하여 지도학습 분류 모델을 학습하였다. 이후 2026년 선수 데이터에는 단순 예측값이 아니라 predict_proba()를 활용해 수상 가능성 점수를 산출하였다. 골든글러브는 포지션별로 수상자가 정해지는 특성이 있으므로, 타자는 각 포지션별 예측 확률 상위 선수를 후보로 선정하였고, 투수는 전체 투수 중 예측 확률 상위 선수를 후보로 선정하였다. 특히 투수 부문은 수상자 라벨 수가 적어 확률값이 매우 낮게 산출되었기 때문에, 확률의 절댓값보다 후보 간 상대 순위를 중심으로 해석하였다.   

지명타자(DH)는 골든글러브 수상 부문에는 포함되지만, 본 분석에 사용한 2026년 타자 데이터의 포지션 컬럼에는 DH가 별도 포지션으로 제공되지 않은 관계로 지명타자 예측의 경우 타자 예측과 따로 진행한다.대신 2026 시즌 예측 단계에서 올스타전 지명타자 후보 명단을 기준으로 DH 후보군을 별도로 구성하고, 기존 타자 예측 모델을 적용하여 후보별 수상 가능성을 비교하였다.

In [130]:
# 2026 타자 데이터 예측

# 원본 복사
hitter_2026_model = hitter_2026.copy()

# 'rank' 컬럼 제거 (예측에 사용되지 않으므로 제거하여 일관성 유지)
if 'rank' in hitter_2026_model.columns:
    hitter_2026_model = hitter_2026_model.drop(columns=['rank'])

# 학습 때 사용한 feature 기준으로 숫자형 변환
for col in hitter_features:
  if col in hitter_2026_model.columns:
    hitter_2026_model[col] = pd.to_numeric(
        hitter_2026_model[col],
        errors="coerce"
    )

# 학습 때 사용한 feature만 추출
x_hitter_2026 = hitter_2026_model[hitter_features]

# 예측 확률 생성
hitter_2026_model["goldenglove_probability"] = (
    hitter_final_model.predict_proba(x_hitter_2026)[:, 1]
)

# 확률 높은 순으로 정렬
hitter_result = hitter_2026_model.sort_values(
    "goldenglove_probability",
    ascending = False
)

hitter_result[[
    'Year', 'Name', 'Team', 'Pos',
    'AVG', 'G', 'PA', 'R', 'H', 'HR', 'RBI',
    'BB', 'SO', 'SLG', 'OBP', 'OPS', 'RISP', 'goldenglove_probability'
]].head(30)

,Year,Name,Team,Pos,AVG,G,PA,R,H,HR,RBI,BB,SO,SLG,OBP,OPS,RISP,goldenglove_probability
4,2026,오스틴,LG,1B,0.349,63,286,53,88,19,59,28,41,0.659,0.420,1.079,0.411,0.426517
0,2026,최원준,KT,OF,0.386,62,297,54,98,5,37,33,40,0.535,0.459,0.994,0.328,0.341639
2,2026,박성한,SSG,SS,0.355,63,280,46,82,3,32,46,27,0.472,0.461,0.933,0.442,0.331860
3,2026,레이예스,롯데,OF,0.352,62,277,28,87,9,44,25,32,0.530,0.412,0.942,0.381,0.255816
5,2026,페라자,한화,OF,0.329,62,287,54,79,13,40,41,54,0.558,0.425,0.983,0.271,0.211789
8,2026,최형우,삼성,OF,0.317,60,256,28,66,8,44,42,34,0.481,0.434,0.915,0.338,0.211240
17,2026,문현빈,한화,OF,0.292,60,280,41,69,9,44,37,30,0.504,0.391,0.895,0.267,0.205266
30,2026,김도영,KIA,3B,0.272,64,276,45,64,19,52,34,44,0.562,0.373,0.935,0.356,0.189766
6,2026,강백호,한화,1B,0.323,58,256,37,73,13,63,28,52,0.566,0.398,0.964,0.435,0.159385
10,2026,박민우,NC,2B,0.306,60,251,39,63,4,37,36,30,0.427,0.420,0.847,0.424,0.143024


In [135]:
# DH 예측
dh_candidates = [
    "문보경",
    "강백호",
    "김선빈",
    "오영수",
    "서건창",
    "손아섭",
    "최형우",
    "전준우",
    "김재환",
    "장성우"
]

# 2026 타자 데이터에서 지타 후보 필터링
# 칼럼 삭제 등 전처리 완료된 hitter_2026_model을 사용
hitter_dh_2026 = hitter_2026_model[hitter_2026_model["Name"].isin(dh_candidates)].copy()

# 타자 확률이 이미 예측되었음으로 포지션을 DH로 표시
hitter_dh_2026["Pos"] = "DH"

# 확률 높은 순으로 정렬
hitter_dh_result = hitter_dh_2026.sort_values(
    "goldenglove_probability", ascending = False
)

hitter_dh_result[[
    'Year', 'Name', 'Team', 'Pos',
    'AVG', 'G', 'PA', 'R', 'H', 'HR', 'RBI',
    'BB', 'SO', 'SLG', 'OBP', 'OPS', 'RISP', 'goldenglove_probability'
]].head(30)


,Year,Name,Team,Pos,AVG,G,PA,R,H,HR,RBI,BB,SO,SLG,OBP,OPS,RISP,goldenglove_probability
8,2026,최형우,삼성,DH,0.317,60,256,28,66,8,44,42,34,0.481,0.434,0.915,0.338,0.211240
6,2026,강백호,한화,DH,0.323,58,256,37,73,13,63,28,52,0.566,0.398,0.964,0.435,0.159385
34,2026,김선빈,KIA,DH,0.267,62,248,26,56,1,22,35,26,0.343,0.375,0.718,0.273,0.063637
47,2026,장성우,KT,DH,0.206,50,199,28,33,8,35,37,49,0.419,0.357,0.776,0.266,0.023890
48,2026,김재환,SSG,DH,0.186,54,222,21,34,8,28,36,67,0.328,0.324,0.652,0.294,0.010017


In [127]:
# 2026 투수 데이터 예측

# 원본 복사
pitcher_2026_model = pitcher_2026.copy()

# 'rank' 컬럼 제거 (예측에 사용되지 않으므로 제거하여 일관성 유지)
if 'rank' in pitcher_2026_model.columns:
    pitcher_2026_model = pitcher_2026_model.drop(columns=['rank'])

# 'HLB' 컬럼을 'HLD'로 변경 (feature list와 일치시키기 위해)
if 'HLB' in pitcher_2026_model.columns:
    pitcher_2026_model = pitcher_2026_model.rename(columns={'HLB': 'HLD'})

# 'IP' 컬럼 전처리 (문자열 fractional innings to float) - 2026 데이터에도 적용
pitcher_2026_model['IP'] = pitcher_2026_model['IP'].apply(convert_ip_to_float)

# 학습 때 사용한 feature 기준으로 숫자형 변환
for col in pitcher_features:
  if col in pitcher_2026_model.columns:
    pitcher_2026_model[col] = pd.to_numeric(
        pitcher_2026_model[col],
        errors="coerce"
    )

# 남은 결측치 중앙값 대체 - 2026 데이터에도 적용
# 훈련 데이터의 IP 중앙값을 사용하거나, 2026 데이터의 중앙값을 사용
if 'IP' in pitcher_features:
    # 훈련 데이터의 IP 중앙값
    ip_median_for_imputation = pitcher_recent_clean['IP'].median()
    pitcher_2026_model['IP'] = pitcher_2026_model['IP'].fillna(ip_median_for_imputation)

# 학습 때 사용한 feature만 추출
x_pitcher_2026 = pitcher_2026_model[pitcher_features]

# 예측 확률 생성
pitcher_2026_model["goldenglove_probability"] = (
    pitcher_final_model.predict_proba(x_pitcher_2026)[:, 1]
)

# 확률 높은 순으로 정렬
pitcher_result = pitcher_2026_model.sort_values(
    "goldenglove_probability",
    ascending = False
)

pitcher_result[[
    'Year', 'Name', 'Team',
    'ERA', 'G', 'W', 'S', 'HD', 'WPCT', 'IP',
    'SO', 'WHIP', 'QS', 'goldenglove_probability'
]].head(30)

,Year,Name,Team,ERA,G,W,S,HD,WPCT,IP,SO,WHIP,QS,goldenglove_probability
1,2026,올러,KIA,2.66,13,7,0,0,0.583,81.333333,86,0.95,8,5.662123e-04
2,2026,류현진,한화,2.84,12,8,0,0,0.800,69.666667,56,1.03,6,4.915260e-04
3,2026,최민석,두산,2.88,12,6,0,0,0.750,68.666667,67,1.24,7,8.530526e-05
12,2026,톨허스트,LG,3.99,13,7,0,0,0.636,34.200000,59,1.20,9,6.277222e-05
4,2026,알칸타라,키움,3.12,12,6,0,0,0.600,34.200000,70,1.15,8,4.701883e-05
9,2026,구창모,NC,3.69,12,6,0,0,0.750,68.333333,56,1.24,7,4.615632e-05
10,2026,임찬규,LG,3.72,12,6,0,0,0.857,65.333333,36,1.55,4,2.111734e-05
11,2026,오러클린,삼성,3.88,13,5,0,0,0.625,67.333333,66,1.23,7,1.874697e-05
15,2026,사우어,KT,4.42,13,5,0,0,0.625,75.333333,58,1.30,6,1.411605e-05
6,2026,곽빈,두산,3.38,12,4,0,0,0.571,66.666667,79,1.38,7,1.331419e-05


In [137]:
# 타자 최종후보

# 타자 예측 확률 기준 정렬
hitter_result = hitter_2026_model.sort_values(
    "goldenglove_probability",
    ascending=False
)

selected_hitters = []

# 포수, 내야수
for pos in ["C", "1B", "2B", "3B", "SS"]:
    temp = hitter_result[hitter_result["Pos"] == pos].head(1)
    if len(temp) > 0:
        selected_hitters.append(temp)

# 외야수
of_top3 = hitter_result[hitter_result["Pos"] == "OF"].head(3)
if len(of_top3) > 0:
    selected_hitters.append(of_top3)

# 지명타자
dh_top1 = (
    hitter_result[hitter_result["Name"].isin(dh_candidates)]
    .sort_values("goldenglove_probability", ascending = False)
    .head(1)
    .copy()
)

# 지명타자 후보가 잡혔을 때만 타자 최종 후보에 추가
if len(dh_top1) > 0:
  dh_top1["Pos"] = "DH"
  selected_hitters.append(dh_top1)
else:
  print("지명타자 후보가 hitter_result에서 매칭되지 않았습니다. dh_candidates의 이름 표기 다시 확인")

final_hitter_candidates = pd.concat(selected_hitters, ignore_index=True)

print("타자 최종 후보: ")
final_hitter_candidates.head(10)

타자 최종 후보: 


,Name,Team,Pos,AVG,G,PA,AB,R,H,2B,3B,HR,TB,RBI,SH,SF,BB,IB,HP,SO,GDP,SLG,OBP,OPS,MH,RISP,PH-BA,Year,goldenglove_probability
0,양의지,두산,C,0.256,61,246,207,22,53,7,0,10,90,34,0,4,22,0,13,31,5,0.435,0.358,0.793,14,0.222,0.500,2026,0.067816
1,오스틴,LG,1B,0.349,63,286,252,53,88,15,3,19,166,59,0,2,28,2,4,41,5,0.659,0.420,1.079,22,0.411,0.000,2026,0.426517
2,박민우,NC,2B,0.306,60,251,206,39,63,13,0,4,88,37,1,2,36,2,6,30,6,0.427,0.420,0.847,16,0.424,0.500,2026,0.143024
3,김도영,KIA,3B,0.272,64,276,235,45,64,9,1,19,132,52,0,2,34,4,5,44,6,0.562,0.373,0.935,14,0.356,0.000,2026,0.189766
4,박성한,SSG,SS,0.355,63,280,231,46,82,16,1,3,109,32,0,2,46,1,1,27,2,0.472,0.461,0.933,24,0.442,0.333,2026,0.331860
5,최원준,KT,OF,0.386,62,297,254,54,98,19,2,5,136,37,3,3,33,0,4,40,2,0.535,0.459,0.994,30,0.328,0.000,2026,0.341639
6,레이예스,롯데,OF,0.352,62,277,247,28,87,15,1,9,131,44,0,3,25,5,2,32,2,0.530,0.412,0.942,25,0.381,0.000,2026,0.255816
7,페라자,한화,OF,0.329,62,287,240,54,79,14,1,13,134,40,2,3,41,4,1,54,6,0.558,0.425,0.983,21,0.271,0.000,2026,0.211789
8,최형우,삼성,DH,0.317,60,256,208,28,66,10,0,8,100,44,0,3,42,2,3,34,7,0.481,0.434,0.915,17,0.338,0.250,2026,0.211240


In [138]:
# 투수 최종 후보

# 투수는 확률값 자체보다 순위 중심으로 해석
pitcher_result = pitcher_2026_model.sort_values(
    "goldenglove_probability",
    ascending=False
)

final_pitcher_candidates = pitcher_result.head(5)

final_pitcher_candidates[[
    "Year", "Name", "Team",
    'ERA', 'G', 'W', 'L', 'S', 'HD', 'WPCT', 'IP',
    'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP',
    'CG', 'SHO', 'QS', 'TBF', 'NP', 'AVG', 'goldenglove_probability'
]]

print("투수 최종 후보: ")
final_pitcher_candidates.head(30)

투수 최종 후보: 


,Name,Team,ERA,G,W,L,S,HD,WPCT,IP,H,HR,BB,HBP,SO,R,ER,WHIP,CG,SHO,QS,BSV,TBF,NP,AVG,2B,3B,SAC,SF,IBB,WP,BK,Year,goldenglove_probability
1,올러,KIA,2.66,13,7,5,0,0,0.583,81.333333,52,5,25,4,86,25,24,0.95,1,1,8,0,320,1211,0.182,5,2,5,1,0,3,0,2026,0.000566
2,류현진,한화,2.84,12,8,2,0,0,0.800,69.666667,62,4,10,2,56,28,22,1.03,0,0,6,0,280,1059,0.239,14,1,7,2,1,1,0,2026,0.000492
3,최민석,두산,2.88,12,6,2,0,0,0.750,68.666667,54,3,31,4,67,28,22,1.24,0,0,7,0,291,1098,0.215,13,0,3,2,0,3,1,2026,0.000085
12,톨허스트,LG,3.99,13,7,4,0,0,0.636,34.200000,65,6,19,7,59,31,31,1.20,0,0,9,0,299,1125,0.243,10,0,2,3,0,1,1,2026,0.000063
4,알칸타라,키움,3.12,12,6,4,0,0,0.600,34.200000,79,11,11,0,70,28,27,1.15,0,0,8,0,318,1154,0.259,12,2,1,1,0,2,0,2026,0.000047


## 2026 골든글러브 후보 예측 보고서

### 1. 분석 목표

본 보고서는 2026년 KBO 골든글러브 수상 가능성이 높은 타자 및 투수 후보를 예측하고, 그 선정 과정을 설명하는 것을 목표로 합니다.

### 2. 데이터 및 모델

*   **학습 데이터:** 2000년부터 2025년까지의 KBO 선수 기록 데이터를 활용하여 `is_goldenglove` (골든글러브 수상 여부) 라벨을 학습했습니다.
*   **예측 대상 데이터:** 2026년 6월 14일 기준 선수 기록 데이터를 사용하여 예측을 수행했습니다.
*   **예측 모델:** 클래스 불균형 문제를 고려하여 타자 및 투수 모두 **로지스틱 회귀(Logistic Regression)** 모델을 최종 모델로 선정하여 `goldenglove_probability` (수상 가능성 확률)을 산출했습니다.

### 3. 골든글러브 후보 선정 기준

#### 3.1 타자 후보 선정

타자는 포지션별로 수상자가 선정되는 특성을 반영하여, 각 포지션 내에서 `goldenglove_probability`가 가장 높은 선수를 최종 후보로 선정했습니다. 외야수의 경우 3명의 수상자를 고려하여 상위 3명을 선정했습니다. 지명타자(DH)는 별도로 정의된 후보군 내에서 가장 높은 확률을 가진 선수를 선정했습니다.

**타자 최종 후보:**


In [139]:
display(final_hitter_candidates)

,Name,Team,Pos,AVG,G,PA,AB,R,H,2B,3B,HR,TB,RBI,SH,SF,BB,IB,HP,SO,GDP,SLG,OBP,OPS,MH,RISP,PH-BA,Year,goldenglove_probability
0,양의지,두산,C,0.256,61,246,207,22,53,7,0,10,90,34,0,4,22,0,13,31,5,0.435,0.358,0.793,14,0.222,0.500,2026,0.067816
1,오스틴,LG,1B,0.349,63,286,252,53,88,15,3,19,166,59,0,2,28,2,4,41,5,0.659,0.420,1.079,22,0.411,0.000,2026,0.426517
2,박민우,NC,2B,0.306,60,251,206,39,63,13,0,4,88,37,1,2,36,2,6,30,6,0.427,0.420,0.847,16,0.424,0.500,2026,0.143024
3,김도영,KIA,3B,0.272,64,276,235,45,64,9,1,19,132,52,0,2,34,4,5,44,6,0.562,0.373,0.935,14,0.356,0.000,2026,0.189766
4,박성한,SSG,SS,0.355,63,280,231,46,82,16,1,3,109,32,0,2,46,1,1,27,2,0.472,0.461,0.933,24,0.442,0.333,2026,0.331860
5,최원준,KT,OF,0.386,62,297,254,54,98,19,2,5,136,37,3,3,33,0,4,40,2,0.535,0.459,0.994,30,0.328,0.000,2026,0.341639
6,레이예스,롯데,OF,0.352,62,277,247,28,87,15,1,9,131,44,0,3,25,5,2,32,2,0.530,0.412,0.942,25,0.381,0.000,2026,0.255816
7,페라자,한화,OF,0.329,62,287,240,54,79,14,1,13,134,40,2,3,41,4,1,54,6,0.558,0.425,0.983,21,0.271,0.000,2026,0.211789
8,최형우,삼성,DH,0.317,60,256,208,28,66,10,0,8,100,44,0,3,42,2,3,34,7,0.481,0.434,0.915,17,0.338,0.250,2026,0.211240


#### 3.2 투수 후보 선정

투수 부문은 학습 데이터 내 골든글러브 수상자 수가 매우 적어 모델의 절대적인 확률 값보다는 상대적인 순위가 더 중요하다고 판단했습니다. 따라서 전체 투수 중에서 `goldenglove_probability`가 가장 높은 **상위 5명**의 선수를 최종 후보로 선정했습니다.

**투수 최종 후보:**


In [140]:
display(final_pitcher_candidates)

,Name,Team,ERA,G,W,L,S,HD,WPCT,IP,H,HR,BB,HBP,SO,R,ER,WHIP,CG,SHO,QS,BSV,TBF,NP,AVG,2B,3B,SAC,SF,IBB,WP,BK,Year,goldenglove_probability
1,올러,KIA,2.66,13,7,5,0,0,0.583,81.333333,52,5,25,4,86,25,24,0.95,1,1,8,0,320,1211,0.182,5,2,5,1,0,3,0,2026,0.000566
2,류현진,한화,2.84,12,8,2,0,0,0.800,69.666667,62,4,10,2,56,28,22,1.03,0,0,6,0,280,1059,0.239,14,1,7,2,1,1,0,2026,0.000492
3,최민석,두산,2.88,12,6,2,0,0,0.750,68.666667,54,3,31,4,67,28,22,1.24,0,0,7,0,291,1098,0.215,13,0,3,2,0,3,1,2026,0.000085
12,톨허스트,LG,3.99,13,7,4,0,0,0.636,34.200000,65,6,19,7,59,31,31,1.20,0,0,9,0,299,1125,0.243,10,0,2,3,0,1,1,2026,0.000063
4,알칸타라,키움,3.12,12,6,4,0,0,0.600,34.200000,79,11,11,0,70,28,27,1.15,0,0,8,0,318,1154,0.259,12,2,1,1,0,2,0,2026,0.000047


### 4. 결론 및 해석

본 분석 결과는 2026 시즌 현재까지의 기록을 바탕으로 한 통계적 예측이며, 실제 골든글러브 선정 기준에는 기록 외적인 요소(팀 기여도, 인기 등)가 포함될 수 있습니다. 특히 투수 부문의 낮은 확률 값은 실제 수상자 수가 적기 때문이며, 이는 후보 간 상대적인 우위를 나타내는 지표로 해석하는 것이 적절합니다.